In [1]:
import os
import glob
import numpy as np
import tensorflow as tf
from PIL import Image
from scipy.io import loadmat
from tensorflow.keras import layers, models
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

path = "Datasets/BSDS500"
tf.keras.utils.set_random_seed(42)


def get_patches(split):
    X = []
    y = []
    files = sorted(glob.glob(path + "/images/" + split + "/*.jpg"))
    for f in files:
        name = os.path.basename(f)[:-4]
        img = np.array(Image.open(f).convert("RGB"))
        d = loadmat(path + "/ground_truth/" + split + "/" + name + ".mat")["groundTruth"]
        b = np.zeros(img.shape[:2])
        for i in range(d.shape[1]):
            b = b + d[0, i]["Boundaries"][0, 0]
        mask = (b > 0).astype(int)[16:-16, 16:-16]
        for label in [0, 1]:
            pts = np.argwhere(mask == label)
            pick = np.random.choice(len(pts), 32, replace=False)
            for r, c in pts[pick]:
                X.append(img[r:r + 32, c:c + 32])
                y.append(label)
    X = preprocess_input(np.array(X, dtype="float32"))
    return X, np.array(y)


Xtr, ytr = get_patches("train")
Xval, yval = get_patches("val")
Xte, yte = get_patches("test")

print("Train patches:", Xtr.shape, " Validation patches:", Xval.shape, " Test patches:", Xte.shape)

base = ResNet50(weights="Pretrained/resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5", include_top=False, pooling="avg", input_shape=(32, 32, 3))
base.trainable = False

model = models.Sequential()
model.add(base)
model.add(layers.Dense(1, activation="sigmoid"))

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(Xtr, ytr, validation_data=(Xval, yval), epochs=5, batch_size=128, verbose=2)

loss, acc = model.evaluate(Xte, yte, verbose=0)
print("Test Accuracy:", round(acc, 4))


Train patches: (12800, 32, 32, 3)  Validation patches: (6400, 32, 32, 3)  Test patches: (12800, 32, 32, 3)
Epoch 1/5
100/100 - 15s - 148ms/step - accuracy: 0.6187 - loss: 0.8794 - val_accuracy: 0.6397 - val_loss: 0.7943
Epoch 2/5
100/100 - 10s - 101ms/step - accuracy: 0.6657 - loss: 0.6785 - val_accuracy: 0.6445 - val_loss: 0.7359
Epoch 3/5
100/100 - 11s - 106ms/step - accuracy: 0.6891 - loss: 0.6181 - val_accuracy: 0.6497 - val_loss: 0.7119
Epoch 4/5
100/100 - 10s - 103ms/step - accuracy: 0.7039 - loss: 0.5859 - val_accuracy: 0.6533 - val_loss: 0.6996
Epoch 5/5
100/100 - 10s - 104ms/step - accuracy: 0.7145 - loss: 0.5661 - val_accuracy: 0.6564 - val_loss: 0.6935
Test Accuracy: 0.6388
